# NeuralProphet for Multivariate Time Series

We quickly test how effective Neural Prophet is at forecasting multivariate time series. 

## Packages

In [ ]:
import requests 
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

from pandas.tseries.holiday import USFederalHolidayCalendar


import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

from neuralprophet import NeuralProphet



## Data Preparation

In [ ]:
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 

rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

BOROUGHS = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']

df = rs.copy()

## Model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from neuralprophet import NeuralProphet

# Boroughs
BOROUGHS = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']

# Ensure datetime
df['ds'] = pd.to_datetime(df['ds'])

# Create full date range
full_dates = pd.date_range('2020-01-01', '2026-02-28', freq='D')
full_index = pd.MultiIndex.from_product([full_dates, BOROUGHS], names=['ds','borough'])

# Reindex and fill missing
df_full = df.set_index(['ds','borough']).reindex(full_index).reset_index()
df_full['y'] = df_full['y'].fillna(0)
df_full = df_full.rename(columns={'borough':'ID'})

m = NeuralProphet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    epochs = 300,
    learning_rate = 0.1
)
m.fit(df_full, freq='D')

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(



Training: 0it [00:00, ?it/s]

,MAE,RMSE,Loss,RegLoss,epoch
0,0.509249,0.643824,0.289357,0.0,0
1,0.185842,0.246057,0.063587,0.0,1
2,0.180381,0.239562,0.060782,0.0,2
3,0.179696,0.238547,0.060363,0.0,3
4,0.179783,0.238628,0.060485,0.0,4
...,...,...,...,...,...
295,0.176097,0.235052,0.058778,0.0,295
296,0.175997,0.234954,0.058771,0.0,296
297,0.176086,0.234659,0.058793,0.0,297
298,0.176019,0.234842,0.058747,0.0,298


In [ ]:
# Forecast horizon
forecast_horizon = 14

# Dictionary to store 14-day forecasts per borough
forecast_by_borough = {}

for b in BOROUGHS:
    df_b = df_full[df_full['ID'] == b].copy()
    
    # Make future dataframe for 14 days
    future_b = m.make_future_dataframe(df_b, periods=forecast_horizon, n_historic_predictions=0)
    
    forecast_b = m.predict(future_b)
    
    forecast_14d_b = forecast_b[forecast_b['ds'] > df_b['ds'].max()][['ds', 'yhat1']].reset_index(drop=True)
    
    forecast_by_borough[b] = forecast_14d_b

Predicting: 88it [00:00, ?it/s]

Predicting: 88it [00:00, ?it/s]

Predicting: 88it [00:00, ?it/s]

Predicting: 88it [00:00, ?it/s]

Predicting: 88it [00:00, ?it/s]

In [ ]:
# Create a copy to work with
data = forecast_by_borough.copy()

for borough in data:
    df = data[borough]
    for i in range(len(df)):
        df.loc[i, 'yhat1'] = round(df.loc[i, 'yhat1'])


wide_df = pd.DataFrame({'ds': data['BRONX']['ds']})
for borough in data:
    wide_df[borough] = data[borough]['yhat1']

wide_df

,ds,BRONX,BROOKLYN,MANHATTAN,QUEENS,STATEN ISLAND
0,2026-03-01,3.0,6.0,4.0,3.0,1.0
1,2026-03-02,7.0,16.0,11.0,7.0,2.0
2,2026-03-03,6.0,15.0,11.0,7.0,2.0
3,2026-03-04,6.0,15.0,10.0,7.0,2.0
4,2026-03-05,6.0,14.0,10.0,7.0,2.0
5,2026-03-06,5.0,12.0,8.0,6.0,1.0
6,2026-03-07,3.0,6.0,4.0,3.0,1.0
7,2026-03-08,3.0,7.0,5.0,3.0,1.0
8,2026-03-09,7.0,16.0,11.0,8.0,2.0
9,2026-03-10,7.0,16.0,11.0,7.0,2.0


# Evaluating the NeuralProphet Model

We at least want to see if the NeuralProphet model is making good predictions.

In [ ]:
import pandas as pd
from neuralprophet import NeuralProphet, set_log_level
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

set_log_level("ERROR")

rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_daily_borough_rs.csv')
rs['created_date'] = pd.to_datetime(rs['created_date'])
rs = rs[(rs['created_date'] >= '2020-01-01') & (rs['created_date'] < '2026-03-01')]
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
BOROUGHS = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']
df = rs.copy()

In [ ]:
full_dates = pd.date_range('2020-01-01', '2026-02-28', freq='D')
full_index = pd.MultiIndex.from_product([full_dates, BOROUGHS], names=['ds','ID'])
df_full = df.set_index(['ds','borough']).reindex(full_index).reset_index()
df_full['y'] = df_full['y'].fillna(0.0)

# use neuralprophet's built in local modeling
m = NeuralProphet(
    trend_global_local="local",      # local trend for each borough
    season_global_local="local",     # local seasonality per borough
    yearly_seasonality=True,
    weekly_seasonality=True,
)

# make a train test split with test size equal to test_days
test_days = 14
df_train, df_test = m.split_df(df_full, valid_p=test_days/len(full_dates), local_split=True)

# fit the model 
metrics = m.fit(df_train, freq="D")

# make future dataframe
future = m.make_future_dataframe(df_test, n_historic_predictions=True)

# make the predictions
forecast = m.predict(future)

# Merge our forecasts with actuals out of test set for evaluation
forecast_eval = forecast.merge(
    df_test[['ds','ID','y']],
    on=['ds','ID'],
    how='inner',  # ensure only rows with matching actuals
    suffixes=('_pred','_true')
)

# metrics to look at
mae = mean_absolute_error(forecast_eval['y_true'], forecast_eval['yhat1'])
rmse = np.sqrt(mean_squared_error(forecast_eval['y_true'], forecast_eval['yhat1']))

print(f"Global‑Local Model MAE: {mae:.3f}, RMSE: {rmse:.3f}")

# print the test results
test_results = m.test(df_test, verbose = True)  
# print(test_results)

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(



Finding best initial lr:   0%|          | 0/251 [00:00<?, ?it/s]

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/lightning_fabric/utilities/cloud_io.py:51: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this

Training: 0it [00:00, ?it/s]

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs

...

Predicting: 88it [00:00, ?it/s]

Predicting: 88it [00:00, ?it/s]

Global‑Local Model MAE: 2.793, RMSE: 3.875


Testing: 0it [00:00, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Runningstage.testing    ┃                           ┃
┃          metric           ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         Loss_test         │    0.03852389380335808    │
│          MAE_val          │    0.12040220201015472    │
│         RMSE_val          │    0.15247446298599243    │
│       RegLoss_test        │            0.0            │
└───────────────────────────┴───────────────────────────┘

In [ ]:
for b in BOROUGHS:
    df_b = forecast_eval[forecast_eval['ID'] == b]
    mae_b = mean_absolute_error(df_b['y_true'], df_b['yhat1'])
    rmse_b = np.sqrt(mean_squared_error(df_b['y_true'], df_b['yhat1']))
    print(f"{b} MAE: {mae_b:.3f}, RMSE: {rmse_b:.3f}")

BRONX MAE: 2.028, RMSE: 2.726
BROOKLYN MAE: 5.080, RMSE: 5.924
MANHATTAN MAE: 3.485, RMSE: 4.428
QUEENS MAE: 2.655, RMSE: 3.492
STATEN ISLAND MAE: 0.718, RMSE: 0.870


In [ ]:
forecast

,ds,y,ID,yhat1,trend,season_yearly,season_weekly
0,2026-02-15,4.0,BRONX,3.098454,7.708558,-1.815376,-2.794729
1,2026-02-16,5.0,BRONX,7.546550,7.703303,-1.800804,1.644051
2,2026-02-17,8.0,BRONX,7.574972,7.698046,-1.784907,1.661833
3,2026-02-18,9.0,BRONX,6.985839,7.692791,-1.767650,1.060699
4,2026-02-19,6.0,BRONX,7.026369,7.687534,-1.749159,1.087994
...,...,...,...,...,...,...,...
70,2026-02-25,0.0,STATEN ISLAND,1.166454,1.396688,-0.583952,0.353718
71,2026-02-26,1.0,STATEN ISLAND,0.994843,1.396142,-0.576777,0.175478
72,2026-02-27,0.0,STATEN ISLAND,0.787132,1.395596,-0.569563,-0.038901
73,2026-02-28,0.0,STATEN ISLAND,0.377379,1.395050,-0.562354,-0.455317


In [ ]:
# Keep only the columns we care about
df_simple = forecast[['ds', 'ID', 'y', 'yhat1']]

# Pivot predictions (yhat1)
y_pred = df_simple.pivot(index='ds', columns='ID', values='yhat1')

# Pivot actuals (keep NaNs so alignment is preserved)
y_actual = df_simple.pivot(index='ds', columns='ID', values='y')

# Combine columns per borough: yhat first, then actual
combined_cols = []
for b in BOROUGHS:
    combined_cols.append((b, 'yhat'))
    combined_cols.append((b, 'actual'))

# Build a new dataframe
df_pivot_list = []
for b in BOROUGHS:
    df_pivot_list.append(y_pred[b].rename(f"{b}_yhat"))
    df_pivot_list.append(y_actual[b].rename(f"{b}_actual"))

df_pivot = pd.concat(df_pivot_list, axis=1)
df_pivot = df_pivot.reset_index()  # bring ds back as a column

# Show result
df_pivot

,ds,BRONX_yhat,BRONX_actual,BROOKLYN_yhat,BROOKLYN_actual,MANHATTAN_yhat,MANHATTAN_actual,QUEENS_yhat,QUEENS_actual,STATEN ISLAND_yhat,STATEN ISLAND_actual
0,2026-02-15,3.098454,4.0,4.066105,8.0,6.101917,9.0,1.074738,7.0,0.253907,2.0
1,2026-02-16,7.546550,5.0,13.860908,11.0,11.657790,7.0,5.914369,5.0,0.995852,0.0
2,2026-02-17,7.574972,8.0,14.378965,9.0,10.731263,12.0,5.596662,8.0,0.970560,0.0
3,2026-02-18,6.985839,9.0,13.531856,22.0,10.838062,13.0,5.270483,8.0,1.119405,0.0
4,2026-02-19,7.026369,6.0,13.369396,19.0,9.982987,10.0,5.030076,7.0,0.948078,1.0
5,2026-02-20,5.968546,5.0,10.292762,14.0,9.306931,12.0,4.302817,5.0,0.739586,0.0
6,2026-02-21,3.290408,4.0,3.527609,14.0,4.884322,13.0,1.688849,6.0,0.330426,0.0
7,2026-02-22,3.189804,2.0,4.591698,9.0,6.387259,8.0,1.373193,2.0,0.301689,1.0
8,2026-02-23,7.646197,2.0,14.409253,3.0,11.988893,3.0,6.210997,2.0,1.043557,0.0
9,2026-02-24,7.680049,4.0,14.942392,12.0,11.097383,7.0,5.887669,2.0,1.017939,1.0


# Residuals Plot